# ⚙️ [1/5] Check your Google Drive

        DiverScan-release
        │
        ├── data
        │        └── yourdata_folder
        │
        ├── projects
        │        └── yourproject_folder
        │                ├── dataset
        │                └── optuna (🟢 `optuna_folder`)
        │                        ├── optuna_summary.txt
        │                        ├── learning_settings.txt
        │                        └── ...
        │
        └── utils

Ensure you have the `optuna_summary.txt` in your optuna folder.

# ⚙️ [2/5] Use a GPU

See the above menu. Select `Runtime` -> `Change runtime type` -> your favorite GPU -> `Save`.

In [ ]:
# Hit left ▶️ of this cell and run it!
CUDA = "cuda:0"

# ⚙️ [3/5] Authorize this notebook

In [ ]:
# Hit left ▶️ of this cell and run it!
# Mounting Google Drive is what Colab needs to reach your files. Elsewhere
# there is nothing to mount, so the notebook carries on with local paths.
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print("Not running in Google Colab; set the folder paths below to local ones.")


# ⚙️ [4/5] Load data

In [ ]:
#@title Hit left ▶️ of this cell and run it!

import os
import sys
from pathlib import Path
import pandas as pd
from IPython.display import display

project_folder = "/content/drive/MyDrive/DiverScan-release/projects/fruitfly" #@param {type:"string"}
optuna_folder_name = "optuna" #@param {type:"string"}

# The project lives under DiverScan/projects; the architecture search wrote
# its results into the optuna folder inside it.
project_folder = Path(project_folder)
optuna_folder = str(project_folder / optuna_folder_name)
os.chdir(optuna_folder)


# Read learning_settings.txt and optuna_summary.txt.
# The code to import is the snapshot 01_build_dataset copied into the project,
# whose folder name is recorded in learning_settings.txt.
loaded_values = {}
with open("learning_settings.txt", "r") as f:
    for line in f:
        key, separator, value = line.rstrip("\n").partition(": ")
        if separator:
            loaded_values[key] = value

utils_snapshot = loaded_values.get("utils_snapshot")
if utils_snapshot:
    utils_folder = str(project_folder / utils_snapshot)
else:
    utils_folder = loaded_values["utils_folder"]
    print("No code snapshot in this project; using", utils_folder)
if utils_folder not in sys.path:
    sys.path.insert(0, utils_folder)

import learning_utils as _learning_utils

settings = _learning_utils.read_learning_settings("learning_settings.txt")
model_type = settings["model_type"]
dataset_folder = str(project_folder / "dataset")
task_type = settings["task_type"]
number_of_attention_branches = settings["number_of_attention_branches"]
seed = settings["seed"]
batch_size = settings["batch_size"]
learning_rate = settings["learning_rate"]
C_CE = settings["C_CE"]
C_S = settings["C_S"]
threshold_of_attention_distribution = settings["threshold_of_attention_distribution"]
threshold_of_loss_penalty = settings["threshold_of_loss_penalty"]
lambda_for_total_variation_regularization = settings["lambda_for_total_variation_regularization"]
conv1d_out_channels = settings["conv1d_out_channels"]
lstm_hidden_size = settings["lstm_hidden_size"]
stgcn_out_channels = settings["stgcn_out_channels"]
node_shuffle = settings["node_shuffle"]
lambda_for_feature_coherence = settings["lambda_for_feature_coherence"]
feature_coherence_threshold_scale = settings["feature_coherence_threshold_scale"]
feature_coherence_sigmoid_sharpness = settings["feature_coherence_sigmoid_sharpness"]

optuna_values = _learning_utils.read_optuna_summary_settings(model_type, "optuna_summary.txt")
dropout_probability = optuna_values["dropout_probability"]
weight_decay = optuna_values["weight_decay"]
temporal_kernel_size_before_branching = optuna_values["temporal_kernel_size_before_branching"]
before_branching_blocks = optuna_values["before_branching_blocks"]
macro_averaged_accuracies = optuna_values["macro_averaged_accuracies"]
similarities = optuna_values["similarities"]

print("\nusing the validation set...")
print("\n⭐️ Optuna best trial ⭐️")

df_single_row_1 = pd.DataFrame(macro_averaged_accuracies)
df_single_row_2 = pd.DataFrame(similarities)
df_combined = pd.concat([df_single_row_1, df_single_row_2], axis=0, ignore_index=True)

df_combined.columns = [f"attention{i+1}" for i in range(len(df_combined.columns))]
df_combined.index = ["macro averaged accuracy", "similarity"]

display(df_combined)


#@markdown ⭐️ Here you can check the summary of `02_search_architecture`, if you did not check `use_preset_hyperparameters`.
#@markdown - `macro averaged accuracy`: Higher values are better. Value 1 means perfectly successful classification, suggesting your model finds difference(s) over conditions. Value 0 means completely failed classification.
#@markdown - `similarity`: Lower values are better. Value 0 means completely different attentions across all branches, suggesting your attention branches find diverse features. Value 1 means completely identical attentions across all branches.

#@markdown ⭐️ If your result has problems...
#@markdown - Number of completed trials on the `02_search_architecture` step might not be sufficient.
#@markdown - Number of files you prepare for `01_build_dataset` might not be sufficient.
#@markdown - There may be no significant differences in the comparison you conducted.


# ⚙️ [5/5] Train your model

1.   Input followings, then select `Runtime` -> `Run all`.

2.   It may take long time, like hours.

3.   If the execution is interrupted, set `resume_from_last_run` to ☑️ then select `Runtime` -> `Run all` again. Do not change other parameters.
⚠️ Please wait at least 30 minutes before resuming to ensure that all files have been saved to Google Drive.

4.   When the connection shuts down after all executions are complete, save and then close the notebook.

5.   Go to the `04_test_model` notebook.

In [ ]:
#@title ⚙️ 1. General settings

resume_from_last_run = True #@param {type:"boolean"}

name_of_folder_to_save_result = "training" #@param {type:"string"}

#@markdown - It could take long time (-hour).
#@markdown - When the connection shuts down after all executions are complete, save and then close the notebook. Go to the `04_test_model` notebook.
#@markdown - ⚠️ If the execution is interrupted, set `resume_from_last_run` to ☑️ then select `Runtime` -> `Run all` again. Do not change other parameters.
#@markdown   - ❗️ Please wait at least 30 minutes before resuming to ensure that all files have been saved to Google Drive.


In [ ]:
#@title ⚙️ 2. Advanced settings
number_of_epochs = 100 #@param {type:"integer"}
save_mode = "save the best epoch" #@param ["save the best epoch", "save every epoch"]
if save_mode == "save the best epoch":
    save_interval_of_epochs = "best"
if save_mode == "save every epoch":
    save_interval_of_epochs = 1

#@markdown - `number_of_epochs` decides length of training.
#@markdown - The .param files of your model are saved during training. The `save the best epoch` mode saves just a .param file only in the best epoch, thus reduces your Google Drive storage usage.

# Codes

In [ ]:
import os

os.environ["PYTHONHASHSEED"] = str(seed)
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
print("CUBLAS_WORKSPACE_CONFIG =", os.environ["CUBLAS_WORKSPACE_CONFIG"])

In [ ]:
from pathlib import Path
import pickle
import gzip
import random
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import copy

In [ ]:
# Make the DiverScan utils importable.
import sys

if utils_folder not in sys.path:
    sys.path.insert(0, utils_folder)

import learning_utils as learning_utils_module

if model_type == "single":
    import single_model as model_module
elif model_type == "multi":
    import multi_model as model_module
else:
    raise ValueError(f'model_type must be "single" or "multi", got {model_type!r}')

# Used to draw the learning curves.
import postprocessing as postprocessing_module

# Used to record the notebook and environment of each run.
import run_record as run_record_module


In [ ]:
# Fix the random seeds, switch on deterministic mode, and pick the device.
# https://pytorch.org/docs/stable/notes/randomness.html
DEVICE = learning_utils_module.setup_determinism(seed, CUDA)


In [ ]:
# Create a folder to save data
project_dir = Path(dataset_folder).parent
save_folder = project_dir / name_of_folder_to_save_result

if not save_folder.exists():
    save_folder.mkdir(parents=True)
elif not resume_from_last_run:  # The folder is exist but you do not resume the last run
    counter = 1
    while save_folder.exists():
        save_folder = project_dir / f"{name_of_folder_to_save_result}_{counter}"
        counter += 1
    save_folder.mkdir(parents=True)

# Change current directory
os.chdir(save_folder)

In [ ]:
# Save settings as a txt file
learning_settings = {
    "optuna_folder": optuna_folder,
    "optuna_folder_name": optuna_folder_name,
}

with open("learning_settings.txt", "a") as file:
    for key, parameter in learning_settings.items():
        file.write(f"{key}: {parameter}\n")

In [ ]:
# Record this notebook and the environment used for this run.
# The notebook file to record, assuming the layout DiverScan/notebooks and
# DiverScan/projects/<project>. In Colab the running notebook is fetched from
# the frontend instead, so the file is only needed outside Colab.
run_record_module.save_run_record(
    utils_folder,
    save_folder,
    notebook_path=project_folder.parent.parent / "notebooks" / "03_train_model.ipynb",
)


In [ ]:
# Change current directory
os.chdir(dataset_folder)

# Make loaders
with gzip.open("dataset_standardized_for_machine_learning.pkl", "rb") as file:
    loaded_data = pickle.load(file)
train_val_test_dictionary = loaded_data["train_val_test_dictionary"]

dataloaders, number_of_classes, number_of_channels, number_of_timestamps, number_of_nodes, number_of_dimensions_of_edge_weight, number_of_channels_of_pose, val_x, val_label, edge_index, val_edge_weight, val_x_pose, has_pose = learning_utils_module.load_data(
    "train_and_val",
    train_val_test_dictionary,
    model_type,
    batch_size,
    DEVICE,
)

train_loader, val_loader = dataloaders

In [ ]:
# Calculate some hyperparameters
threshold_of_loss_penalty, lambda_for_total_variation_regularization, penalty_weight_for_attention_distribution = learning_utils_module.calculate_hyperparameters(
    threshold_of_loss_penalty,
    lambda_for_total_variation_regularization,
    C_CE,
    C_S,
    number_of_classes,
    number_of_attention_branches,
)

# Bundle the loss hyperparameters and the validation tensors into config objects.
loss_config = learning_utils_module.LossConfig(
    task_type=task_type,
    number_of_attention_branches=number_of_attention_branches,
    C_CE=C_CE,
    C_S=C_S,
    threshold_of_loss_penalty=threshold_of_loss_penalty,
    lambda_for_total_variation_regularization=lambda_for_total_variation_regularization,
    penalty_weight_for_attention_distribution=penalty_weight_for_attention_distribution,
    threshold_of_attention_distribution=threshold_of_attention_distribution,
    lambda_for_feature_coherence=lambda_for_feature_coherence,
    feature_coherence_threshold_scale=feature_coherence_threshold_scale,
    feature_coherence_sigmoid_sharpness=feature_coherence_sigmoid_sharpness,
)

val_data = learning_utils_module.ValidationData(
    x=val_x,
    label=val_label,
    edge_weight=val_edge_weight,
    x_pose=val_x_pose,
)

model_spec = learning_utils_module.ModelSpec(
    model_type=model_type,
    number_of_classes=number_of_classes,
    number_of_channels=number_of_channels,
    number_of_timestamps=number_of_timestamps,
    number_of_attention_branches=number_of_attention_branches,
    number_of_channels_of_pose=number_of_channels_of_pose,
    number_of_nodes=number_of_nodes,
    number_of_dimensions_of_edge_weight=number_of_dimensions_of_edge_weight,
    edge_index=edge_index,
    conv1d_out_channels=conv1d_out_channels,
    lstm_hidden_size=lstm_hidden_size,
    stgcn_out_channels=stgcn_out_channels,
    node_shuffle=node_shuffle,
)


In [ ]:
# Change current directory
os.chdir(optuna_folder)

# Read branch information from optuna_summary.txt
with open("optuna_summary.txt", "r") as file:
    optuna_summary = file.read()
branch_list = learning_utils_module.read_optuna_summary(optuna_summary)

In [ ]:
# Generate configs
before_branch_config, branch_config = learning_utils_module.build_configs(
    model_spec,
    dropout_probability,
    before_branching_blocks,
    temporal_kernel_size_before_branching,
    branch_list,
)


In [ ]:
# Load the model
model = learning_utils_module.build_model(
    model_module,
    model_spec,
    before_branch_config,
    branch_config,
    DEVICE,
)

print(model)


# Set up the loss function and optimizer.
criterion = nn.CrossEntropyLoss().to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)

In [ ]:
# Parameters of the model
print("state_dict of the model:")
number_of_total_parameters = 0
for parameter_tensor in model.state_dict():
    print(parameter_tensor, "\t", model.state_dict()[parameter_tensor].shape, "\t", model.state_dict()[parameter_tensor].device)
    number_of_total_parameters += np.prod(model.state_dict()[parameter_tensor].shape)  # Accumulating the number of parameters across all parameter tensors.
print("Number of total parameters of the model: ", number_of_total_parameters)

# Parameters of the optimizer
print("Optimizer\"s state_dict:")
for variable_name in optimizer.state_dict():
    print(variable_name, "\t", optimizer.state_dict()[variable_name])

In [ ]:
# Change current directory
os.chdir(save_folder)


# Load the checkpoint of training
if resume_from_last_run:
    if os.path.exists("latest_epoch.pth"):
        checkpoint = torch.load("latest_epoch.pth")
        model.load_state_dict(checkpoint["model_state_dict"])
        optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
        start_epoch = checkpoint["epoch"] + 1
        print(f"Resuming from epoch {start_epoch}")
    else:
        print("No checkpoints found. Starting from scratch.")
        start_epoch = 0
else:
    start_epoch = 0
    print("start_epoch =", start_epoch)

# Training log

In [ ]:
#@title Log

# start = torch.cuda.Event(enable_timing=True)
# end = torch.cuda.Event(enable_timing=True)
# start.record()


learning_utils_module.train_and_validate(
    model_type,
    model,
    train_loader,
    val_loader,
    val_data,
    has_pose,
    criterion,
    optimizer,
    start_epoch,
    number_of_epochs,
    loss_config,
    trial=None,  # This notebook is not for Optuna trial
    save_interval_of_epochs=save_interval_of_epochs,
)

postprocessing_module.draw_all_loss_curves(learning_utils_module)


# end.record()
# torch.cuda.synchronize()
# elapsed_time = start.elapsed_time(end)
# print(elapsed_time / 1000, "sec.")






try:
    from google.colab import runtime
    import time
    time.sleep(5)
    runtime.unassign()
    print("The runtime has been unassigned.")
except ImportError:
    # Not running in Google Colab environment
    pass